# 🛡️ Threat Detection via Graph + RAG + LLM — Walkthrough

This notebook demonstrates the full pipeline:
1. Connect to Neo4j
2. Explore the graph schema
3. Inject synthetic attacks
4. Run MITRE ATT&CK enrichment
5. Execute detection queries
6. Visualize threat subgraphs
7. Generate an AI analyst report via RAG

## 1. Setup & Connection

In [ ]:
import sys
sys.path.insert(0, '.')

from config.neo4j_connection import get_session
from config.schema_setup import setup_schema

# Verify connection
with get_session() as session:
    result = session.run("RETURN 'Connected to Neo4j!' AS msg")
    print(result.single()['msg'])

## 2. Schema Setup

Creates constraints and indexes for our graph schema:
- **Nodes**: User, Device, IP, Malware, Technique, Campaign
- **Relationships**: AUTHENTICATED_TO, COMMUNICATED_WITH, MATCHES_MALWARE, USES_TECHNIQUE

In [ ]:
setup_schema()

## 3. Inject Synthetic Attacks

We inject 4 types of attack scenarios into the graph:

| Attack | Description |
|---|---|
| Impossible Travel | Same user in 2 countries within minutes |
| Credential Stuffing | 50 failed logins from one IP |
| Lateral Movement | User hops across 6 devices |
| Data Exfiltration | Large outbound transfer to external IP |

In [ ]:
from synthetic_injection.inject_attacks import inject_all_attacks

inject_all_attacks(
    impossible_travel=5,
    credential_stuffing=3,
    lateral_movement=4,
    data_exfiltration=5
)

## 4. Explore the Graph

Let's see what's in our graph now.

In [ ]:
from detection.cypher_queries import get_graph_summary
import pandas as pd

with get_session() as session:
    summary = get_graph_summary(session)

df_summary = pd.DataFrame(list(summary.items()), columns=['Node Type', 'Count'])
df_summary.sort_values('Count', ascending=False)

## 5. MITRE ATT&CK Enrichment

Map each attack type to real MITRE ATT&CK technique IDs.

In [ ]:
from data_ingestion.mitre_enrichment import run_mitre_enrichment, get_technique_summary

run_mitre_enrichment()

# Show the mapping
with get_session() as session:
    mitre_data = get_technique_summary(session)

df_mitre = pd.DataFrame(mitre_data)
df_mitre

## 6. Run Detection Queries

Execute all 4 Cypher-based detection queries.

In [ ]:
from detection.cypher_queries import run_all_detections

findings, summary = run_all_detections()

for attack_type, alerts in findings.items():
    print(f"\n{'='*60}")
    print(f"{attack_type.upper()}: {len(alerts)} alerts")
    print(f"{'='*60}")
    for alert in alerts[:2]:
        for k, v in alert.items():
            print(f"  {k}: {v}")
        print()

## 7. Visualize Threat Graph

Build an interactive graph from the detection findings.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Build a quick visualization of lateral movement
G = nx.DiGraph()

lat_alerts = findings.get('lateral_movement', [])
for alert in lat_alerts[:3]:
    user = alert['user_id']
    devices = alert.get('device_chain', [])
    G.add_node(user, node_type='user')
    for i, dev in enumerate(devices):
        G.add_node(dev, node_type='device')
        if i == 0:
            G.add_edge(user, dev)
        if i > 0:
            G.add_edge(devices[i-1], dev)

color_map = ['#4CAF50' if G.nodes[n].get('node_type') == 'user' else '#AB47BC' for n in G.nodes]

plt.figure(figsize=(14, 8))
pos = nx.spring_layout(G, k=2, seed=42)
nx.draw(G, pos, with_labels=True, node_color=color_map, 
        node_size=800, font_size=8, font_weight='bold',
        edge_color='#666', arrows=True, arrowsize=15)
plt.title('Lateral Movement — User → Device Chains', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize data exfiltration
G_exfil = nx.DiGraph()

exfil_alerts = findings.get('data_exfiltration', [])
for alert in exfil_alerts:
    dev = alert['source_device']
    ip = alert['destination_ip']
    mb = alert.get('megabytes', 0)
    G_exfil.add_node(dev, node_type='device')
    G_exfil.add_node(ip, node_type='ip')
    G_exfil.add_edge(dev, ip, weight=mb)

color_map_exfil = ['#78909C' if G_exfil.nodes[n].get('node_type') == 'device' else '#42A5F5' for n in G_exfil.nodes]

plt.figure(figsize=(12, 6))
pos = nx.spring_layout(G_exfil, k=1.5, seed=42)
nx.draw(G_exfil, pos, with_labels=True, node_color=color_map_exfil,
        node_size=1000, font_size=8, font_weight='bold',
        edge_color='#FF6B6B', arrows=True, arrowsize=15, width=2)
plt.title('Data Exfiltration — Device → External IP', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Generate AI Analyst Report (RAG)

Send all findings to GPT-4o-mini via the RAG pipeline for a comprehensive threat report.

> **Note**: Requires a valid `OPENAI_API_KEY` in `~/.env`

In [ ]:
from rag_pipeline.generate_report import run_full_pipeline
from IPython.display import Markdown

report = run_full_pipeline()
Markdown(report)

## 9. Cleanup (Optional)

Remove all synthetic attack data from the graph.

In [ ]:
# Uncomment to clean up:
# from synthetic_injection.inject_attacks import remove_synthetic_attacks
# with get_session() as session:
#     remove_synthetic_attacks(session)
# print('Cleaned up!')